# Curve Bootstrapping, Swap Pricing & Sensitivity Analysis

This notebook:
1. Bootstraps **SOFR**, **TermSOFR3m**, **ICP**, and **CLP/USD collateral** curves
2. Prices a **5Y SOFR IRS**, a **TermSOFR3m basis swap**, and a **CLP ICP OIS swap**
3. Computes the **CLP/USD cross-currency basis** from the collateral curve
4. Computes **DV01 / sensitivity** per pillar by bumping exposed `SimpleQuote` objects

In [25]:
import json
from curveengine import CurveEngine
import ORE as ore
import pandas as pd

In [26]:
def qs_sensitivities_for(product_label):
    """Return a dict  pillar_name -> dv01  for the given QS product."""
    for p in qs_results['products']:
        if p['label'] == product_label:
            return {s['pillar']: s['dv01'] for s in p['sensitivities']}
    return {}

def classify_pillar(pillar):
    """Return (curve_bucket, tenor) for a QS pillar name."""
    if pillar == 'USD/CLP':
        return ('FX', 'USD/CLP')
    parts = pillar.rsplit('_', 1)
    tenor = parts[-1] if len(parts) > 1 else pillar
    if 'TermSOFR3m' in pillar:
        return ('TermSOFR3m', tenor)
    if 'FxForwardPoints' in pillar or 'CrossCurrencySwap' in pillar:
        return ('CLP_COLLUSD', tenor)
    if 'ICP' in pillar:
        return ('ICP', tenor)
    if 'SOFR' in pillar:
        return ('SOFR', tenor)
    return ('Other', tenor)

def build_qs_lookup(qs_sens):
    """Build dict (curve_bucket, tenor) -> dv01 from QS sensitivities."""
    lookup = {}
    for pillar, dv01 in qs_sens.items():
        curve, tenor = classify_pillar(pillar)
        lookup[(curve, tenor)] = dv01
    return lookup

def compare_sensitivities(ore_rows, qs_label, fx_divisor=1.0):
    """Merge ORE bump DV01s with QS AD DV01s and return a comparison DataFrame."""
    qs_sens = qs_sensitivities_for(qs_label)
    qs_lookup = build_qs_lookup(qs_sens)
    for row in ore_rows:
        curve = row.get('Curve', row.get('curve_bucket', ''))
        tenor = row['Tenor']
        qs_dv01 = qs_lookup.get((curve, tenor), 0.0)
        row['DV01_QS'] = round(qs_dv01, 4)
        row['Diff'] = round(row['DV01 (USD/bp)'] - qs_dv01, 4)
    return pd.DataFrame(ore_rows)

## Load Configuration & Bootstrap All Curves

Market data loaded from `sensitivity_config.json`.

In [27]:
with open("sensitivity_config.json") as f:
    config = json.load(f)

engine = CurveEngine(config)

# Retrieve curves and indexes
curve_names = ["SOFR", "TermSOFR3m", "ICP", "Collateral(CLP/USD)"]
curves = {}
for name in curve_names:
    curves[name] = {
        'curve': engine.getCurve(name),
        'index': engine.getIndex(name),
        'quotes': engine.getQuotes(name)
    }

refDate = ore.Date(11, 11, 2025)
ore.Settings.instance().evaluationDate = refDate
print(f"Curves built   : SOFR, TermSOFR3m, ICP, Collateral(CLP/USD)")

Curves built   : SOFR, TermSOFR3m, ICP, Collateral(CLP/USD)


In [28]:
with open('qs_results.json') as file:
    qs_results = json.load(file)

In [29]:
ore_data = {}
dc = ore.Actual360()
for key,vals in curves.items():
    curve = vals['curve']
    nodes = curve.nodes()
    ore_data[key] = {
        'date': [d.ISO() for d,_ in nodes],
        'yfs':[dc.yearFraction(refDate, d) for d,_ in nodes],
        'dfs': [d for _,d in nodes]
    }

qs_data = {}
dc = ore.Actual360()
for curve in qs_results['curves']:
    nodes = curve['nodes']
    qs_data[curve['name'] ] = {
        'date': [d['date'] for d in nodes],
        'yfs':[d['year_fraction'] for d in nodes],
        'dfs': [d['discount_factor'] for d in nodes],
    }

compare = {}
for k in ore_data.keys():
    ore_df = pd.DataFrame(ore_data[k])
    qs_df = pd.DataFrame(qs_data[k])
    compare[k] = ore_df.merge(qs_df, right_on='date', left_on='date', how='outer', suffixes=['_ore','_qs'])
    compare[k]['diff'] = compare[k]['dfs_ore'] - compare[k]['dfs_qs']

In [30]:
for k in compare.keys():
    print(f"Curve: {k}")
    display(compare[k])

Curve: SOFR


,date,yfs_ore,dfs_ore,yfs_qs,dfs_qs,diff
0,2025-11-11,0.000000,1.000000,0.000000,1.000000,0.000000e+00
1,2025-11-12,0.002778,0.999880,0.002778,0.999880,1.110223e-16
2,2026-02-11,0.255556,0.989043,0.255556,0.989043,-1.110223e-16
3,2026-05-11,0.502778,0.979079,0.502778,0.979079,-6.772360e-15
4,2026-11-11,1.013889,0.959706,1.013889,0.959706,-7.505108e-14
5,2027-11-11,2.027778,0.924388,2.027778,0.924388,2.109424e-15
6,2028-11-11,3.044444,0.890829,3.044444,0.890829,-4.762857e-14
7,2029-11-11,4.058333,0.858579,4.058333,0.858579,2.331468e-15
8,2030-11-11,5.072222,0.827320,5.072222,0.827320,3.885781e-15
9,2032-11-11,7.102778,0.767340,7.102778,0.767340,3.552714e-15


Curve: TermSOFR3m


,date,yfs_ore,dfs_ore,yfs_qs,dfs_qs,diff
0,2025-11-11,0.000000,1.000000,0.000000,1.000000,0.000000e+00
1,2026-02-11,0.255556,0.988968,0.255556,0.988968,1.110223e-16
2,2026-05-11,0.502778,0.979017,0.502778,0.979017,-1.898481e-14
3,2026-11-11,1.013889,0.959552,1.013889,0.959552,-2.220446e-15
4,2027-11-11,2.027778,0.924023,2.027778,0.924023,-2.220446e-16
5,2028-11-11,3.044444,0.890223,3.044444,0.890223,-4.962697e-14
6,2029-11-11,4.058333,0.857732,4.058333,0.857732,2.220446e-16
7,2030-11-11,5.072222,0.826214,5.072222,0.826214,1.665335e-15
8,2032-11-11,7.102778,0.765696,7.102778,0.765696,1.887379e-15
9,2035-11-11,10.144444,0.681022,10.144444,0.681022,1.506573e-13


Curve: ICP


,date,yfs_ore,dfs_ore,yfs_qs,dfs_qs,diff
0,2025-11-11,0.000000,1.000000,0.000000,1.000000,0.000000e+00
1,2025-11-12,0.002778,0.999861,0.002778,0.999861,0.000000e+00
2,2026-02-11,0.255556,0.987433,0.255556,0.987433,0.000000e+00
3,2026-05-11,0.502778,0.975717,0.502778,0.975717,-7.771561e-16
4,2026-11-11,1.013889,0.952590,1.013889,0.952590,1.965095e-14
5,2027-11-11,2.027778,0.910179,2.027778,0.910179,-7.771561e-16
6,2028-11-11,3.044444,0.872257,3.044444,0.872257,-3.441691e-15
7,2030-11-11,5.072222,0.802576,5.072222,0.802576,-1.550982e-13
8,2032-11-11,7.102778,0.740443,7.102778,0.740443,-9.903189e-14
9,2035-11-11,10.144444,0.654370,10.144444,0.654370,-1.743050e-14


Curve: Collateral(CLP/USD)


,date,yfs_ore,dfs_ore,yfs_qs,dfs_qs,diff
0,2025-11-11,0.000000,1.000000,0.000000,1.000000,0.000000e+00
1,2025-12-11,0.083333,0.994817,0.083333,0.994817,-2.742251e-14
2,2026-02-11,0.255556,0.983468,0.255556,0.983468,-3.330669e-16
3,2026-05-11,0.502778,0.965857,0.502778,0.965857,-6.772360e-15
4,2026-11-11,1.013889,0.950459,1.013889,0.950459,5.928591e-14
5,2027-11-11,2.027778,0.907731,2.027778,0.907731,3.774758e-15
6,2028-11-11,3.044444,0.869967,3.044444,0.869967,9.547918e-15
7,2030-11-11,5.072222,0.796764,5.072222,0.796764,5.107026e-15
8,2032-11-11,7.102778,0.730045,7.102778,0.730045,-5.551115e-14
9,2035-11-11,10.144444,0.645087,10.144444,0.645087,1.632028e-14


In [31]:
calendar   = ore.NullCalendar()
convention = ore.Following
notional   = 10_000_000.0
start      = calendar.advance(refDate, ore.Period("2D"))
maturity   = calendar.advance(start, ore.Period("5Y"))

sofr_curve = curves['SOFR']['curve']
sofr_index = curves['SOFR']['index']
fixed_sched = ore.Schedule(start, maturity, ore.Period(ore.Semiannual),
                           calendar, convention, convention,
                           ore.DateGeneration.Forward, False)
float_sched = ore.Schedule(start, maturity, ore.Period(ore.Quarterly),
                           calendar, convention, convention,
                           ore.DateGeneration.Forward, False)

sofr_swap = ore.VanillaSwap(ore.VanillaSwap.Receiver, notional,
                             fixed_sched, 0.0378, dc,
                             float_sched, sofr_index, 0.0, dc)

sofr_handle = ore.YieldTermStructureHandle(sofr_curve)
sofr_swap.setPricingEngine(ore.DiscountingSwapEngine(sofr_handle))

print(f"SOFR OIS 5Y  |  NPV = {sofr_swap.NPV()} USD  |  Fair Rate = {sofr_swap.fairRate():.6%}")

SOFR OIS 5Y  |  NPV = 342.59551083738916 USD  |  Fair Rate = 3.779250%


In [32]:
sofr_tenors  = ["1D","3M","6M","1Y","2Y","3Y","4Y","5Y","7Y","10Y","15Y","20Y","30Y"]
sofr_quotes  = curves['SOFR']['quotes']
bump = 1e-4
sofr_swap_rows = []
for tenor, q in zip(sofr_tenors, sofr_quotes):
    rq = q['rate']
    orig = rq.value()
    rq.setValue(orig + bump)
    v_up = sofr_swap.NPV()
    rq.setValue(orig - bump)
    v_down = sofr_swap.NPV()
    rq.setValue(orig)
    dv01 = (v_up - v_down) / 2
    sofr_swap_rows.append({"Curve": "SOFR", "Tenor": tenor, "Quote": round(orig, 6), "DV01 (USD/bp)": round(dv01, 2)})
compare_sensitivities(sofr_swap_rows, "SOFR OIS 5Y Swap")

,Curve,Tenor,Quote,DV01 (USD/bp),DV01_QS,Diff
0,SOFR,1D,0.04330,2.75,2.7463,0.0037
1,SOFR,3M,0.04335,2.78,2.7769,0.0031
2,SOFR,6M,0.04250,0.10,0.1015,-0.0015
3,SOFR,1Y,0.04100,0.00,0.0030,-0.0030
4,SOFR,2Y,0.03920,-0.00,-0.0006,0.0006
5,SOFR,3Y,0.03840,-0.00,-0.0007,0.0007
6,SOFR,4Y,0.03800,-0.00,-0.0009,0.0009
7,SOFR,5Y,0.03780,-4555.18,-4555.1788,-0.0012
8,SOFR,7Y,0.03770,-17.69,-17.6898,-0.0002
9,SOFR,10Y,0.03790,0.00,0.0000,0.0000


In [33]:
ts3m_index = curves['TermSOFR3m']['index']
ts3m_swap = ore.VanillaSwap(ore.VanillaSwap.Receiver, notional,
                             fixed_sched, 0.04, dc,
                             float_sched, ts3m_index, 0.0, dc)
ts3m_swap.setPricingEngine(ore.DiscountingSwapEngine(sofr_handle))

print(f"TermSOFR3m 5Y  |  NPV = {ts3m_swap.NPV():,.2f} USD  |  Fair Rate = {ts3m_swap.fairRate():.6%}")

TermSOFR3m 5Y  |  NPV = 88,754.68 USD  |  Fair Rate = 3.805675%


In [34]:
ts3m_tenors  = ["3M","6M","1Y","2Y","3Y","4Y","5Y","7Y","10Y","15Y","20Y","30Y"]
ts3m_quotes  = curves['TermSOFR3m']['quotes']

ts3m_swap_rows = []

# Bump TermSOFR3m (forecast curve)
for tenor, q in zip(ts3m_tenors, ts3m_quotes):
    rq = q.get('spread', q.get('rate'))
    orig = rq.value()
    rq.setValue(orig + bump)
    v_up = ts3m_swap.NPV()
    rq.setValue(orig - bump)
    v_down = ts3m_swap.NPV()
    rq.setValue(orig)
    dv01 = (v_up - v_down) / 2
    ts3m_swap_rows.append({"Curve": "TermSOFR3m", "Tenor": tenor, "DV01 (USD/bp)": round(dv01, 2)})

# Bump SOFR (discount curve)
for tenor, q in zip(sofr_tenors, sofr_quotes):
    rq = q['rate']
    orig = rq.value()
    rq.setValue(orig + bump)
    v_up = ts3m_swap.NPV()
    rq.setValue(orig - bump)
    v_down = ts3m_swap.NPV()
    rq.setValue(orig)
    dv01 = (v_up - v_down) / 2
    ts3m_swap_rows.append({"Curve": "SOFR", "Tenor": tenor, "DV01 (USD/bp)": round(dv01, 2)})

compare_sensitivities(ts3m_swap_rows, "TermSOFR3m 5Y Swap")

,Curve,Tenor,DV01 (USD/bp),DV01_QS,Diff
0,TermSOFR3m,3M,5.56,5.5605,-0.0005
1,TermSOFR3m,6M,-0.01,-0.0112,0.0012
2,TermSOFR3m,1Y,0.00,0.0021,-0.0021
3,TermSOFR3m,2Y,-0.00,-0.0043,0.0043
4,TermSOFR3m,3Y,-0.00,-0.0022,0.0022
5,TermSOFR3m,4Y,-0.00,-0.0011,0.0011
6,TermSOFR3m,5Y,-4576.89,-4576.8884,-0.0016
7,TermSOFR3m,7Y,-17.69,-17.6894,-0.0006
8,TermSOFR3m,10Y,0.00,0.0000,0.0000
9,TermSOFR3m,15Y,0.00,0.0000,0.0000


In [35]:
clp_notional = 5_000_000_000.0
clp_fixed_rate = 0.0440
fx_spot     = 935.0

icp_index = curves['ICP']['index']
coll_curve = curves['Collateral(CLP/USD)']['curve']
clp_fixed_sched = ore.Schedule(start, maturity, ore.Period(ore.Semiannual),
                               calendar, convention, convention,
                               ore.DateGeneration.Forward, False)
clp_float_sched = ore.Schedule(start, maturity, ore.Period(ore.Quarterly),
                               calendar, convention, convention,
                               ore.DateGeneration.Forward, False)

icp_swap = ore.VanillaSwap(ore.VanillaSwap.Receiver, clp_notional,
                            clp_fixed_sched, clp_fixed_rate, dc,
                            clp_float_sched, icp_index, 0.0, dc)

# Discount with CLP/USD collateral curve
coll_handle = ore.YieldTermStructureHandle(coll_curve)
icp_swap.setPricingEngine(ore.DiscountingSwapEngine(coll_handle))

print(f"CLP ICP OIS 5Y  |  NPV = {icp_swap.NPV()/fx_spot:,.2f} CLP")

CLP ICP OIS 5Y  |  NPV = 4.44 CLP


In [37]:
fx_spot = 935.0
icp_tenors  = ["1D","3M","6M","1Y","2Y","3Y","5Y","7Y","10Y","20Y"]
icp_quotes  = curves['ICP']['quotes']
coll_tenors = ["1M","3M","6M","1Y","2Y","3Y","5Y","7Y","10Y","20Y"]
coll_quotes = curves['Collateral(CLP/USD)']['quotes']

icp_swap_rows = []

# Bump ICP (forecast curve)
for tenor, q in zip(icp_tenors, icp_quotes):
    rq = q['rate']
    orig = rq.value()
    rq.setValue(orig + bump)
    v_up = icp_swap.NPV()
    rq.setValue(orig - bump)
    v_down = icp_swap.NPV()
    rq.setValue(orig)
    dv01 = (v_up - v_down) / 2 / fx_spot
    icp_swap_rows.append({"Curve": "ICP", "Tenor": tenor, "DV01 (USD/bp)": round(dv01, 2)})

# Bump CLP_COLLUSD (collateral / discount curve)
for tenor, q in zip(coll_tenors, coll_quotes):
    if 'fxPoints' in q:
        rq = q['fxPoints']
    elif 'rate' in q:
        rq = q['rate']
    orig = rq.value()
    rq.setValue(orig + bump)
    v_up = icp_swap.NPV()
    rq.setValue(orig - bump)
    v_down = icp_swap.NPV()
    rq.setValue(orig)
    dv01 = (v_up - v_down) / 2 / fx_spot
    icp_swap_rows.append({"Curve": "CLP_COLLUSD", "Tenor": tenor, "DV01 (USD/bp)": round(dv01, 2)})

# Bump SOFR (affects collateral curve via xccy helpers)
for tenor, q in zip(sofr_tenors, sofr_quotes):
    rq = q['rate']
    orig = rq.value()
    rq.setValue(orig + bump)
    v_up = icp_swap.NPV()
    rq.setValue(orig - bump)
    v_down = icp_swap.NPV()
    rq.setValue(orig)
    dv01 = (v_up - v_down) / 2 / fx_spot
    icp_swap_rows.append({"Curve": "SOFR", "Tenor": tenor, "DV01 (USD/bp)": round(dv01, 2)})

compare_sensitivities(icp_swap_rows, "CLP ICP OIS 5Y Swap (USD collateral)")

,Curve,Tenor,DV01 (USD/bp),DV01_QS,Diff
0,ICP,1D,1.46,1.4624,-0.0024
1,ICP,3M,-0.94,-0.9371,-0.0029
2,ICP,6M,1.13,1.1301,-0.0001
3,ICP,1Y,1.24,1.2370,0.0030
4,ICP,2Y,0.31,0.3068,0.0032
5,ICP,3Y,0.62,0.6161,0.0039
6,ICP,5Y,-2370.12,-2370.1188,-0.0012
7,ICP,7Y,-9.18,-9.1761,-0.0039
8,ICP,10Y,0.00,0.0000,0.0000
9,ICP,20Y,0.00,0.0000,0.0000


In [38]:
usd_not     = 10_000_000.0
clp_not     = usd_not * fx_spot
xccy_spread = 0.0050  # 50 bp on CLP (receive) leg

icp_curve = curves['ICP']['curve']
xccy_sched = ore.Schedule(start, maturity, ore.Period(ore.Quarterly),
                          calendar, convention, convention,
                          ore.DateGeneration.Forward, False)

# Pay USD SOFR flat, Receive CLP ICP + 50 bp
xccy_swap = ore.CrossCcyBasisSwap(
    usd_not,  ore.USDCurrency(), xccy_sched, sofr_index, 0.0, 1.0,
    clp_not,  ore.CLPCurrency(), xccy_sched, icp_index, xccy_spread, 1.0,
)

coll_handle = ore.YieldTermStructureHandle(coll_curve)
xccy_engine = ore.CrossCcySwapEngine(
    ore.USDCurrency(), sofr_handle,
    ore.CLPCurrency(), coll_handle,
    ore.QuoteHandle(ore.SimpleQuote(1/fx_spot))
)
xccy_swap.setPricingEngine(xccy_engine)

print(f"CLP/USD XCCY Basis 5Y  |  NPV = {xccy_swap.NPV():,.2f}")
print(f"  Pay USD leg NPV = {xccy_swap.legNPV(0):,.2f}")
print(f"  Rec CLP leg NPV = {xccy_swap.legNPV(1):,.2f}")
print(f"  Fair Spread on CLP leg = {xccy_swap.fairPaySpread()*10000:.2f} bp")

CLP/USD XCCY Basis 5Y  |  NPV = 158,661.26
  Pay USD leg NPV = -0.00
  Rec CLP leg NPV = 158,661.26
  Fair Spread on CLP leg = 34.57 bp


In [39]:

xccy_label = "Cross-Currency Swap CLP/USD 5Y (receive CLP ICP, pay USD SOFR)"
qs_cfs = next(p for p in qs_results['products'] if p['label'] == xccy_label)['cashflows']

for leg_idx, (ccy, side_label) in enumerate([("USD", "Pay SOFR"), ("CLP", "Receive ICP")]):
    ore_rows = sorted(
        [{"payment_date": c.date().ISO(), "amount_ore": c.amount()} for c in xccy_swap.leg(leg_idx)],
        key=lambda r: r["payment_date"],
    )
    qs_rows = sorted(
        [{"payment_date": cf["payment_date"], "amount_qs": cf["amount"],
          "type": cf["cashflow_type"], "rate_qs": cf.get("rate")}
         for cf in qs_cfs if cf["currency"] == ccy],
        key=lambda r: r["payment_date"],
    )

    df_ore = pd.DataFrame(ore_rows).reset_index(drop=True)
    df_qs  = pd.DataFrame(qs_rows).reset_index(drop=True)

    df = pd.concat([df_qs[["payment_date", "type", "rate_qs", "amount_qs"]],
                     df_ore[["amount_ore"]]], axis=1)
    df["diff"] = df["amount_ore"] - df["amount_qs"]
    df["diff_%"] = (df["diff"] / df["amount_qs"].abs()) * 100

    print(f"\n=== {ccy} Leg ({side_label}) ===")
    display(df)


=== USD Leg (Pay SOFR) ===


,payment_date,type,rate_qs,amount_qs,amount_ore,diff,diff_%
0,2025-11-13,Disbursement,NaN,1.000000e+07,-1.000000e+07,-2.000000e+07,-2.000000e+02
1,2026-02-13,FloatingRateCoupon,0.043301,1.106572e+05,1.106572e+05,-1.332955e-08,-1.204581e-11
2,2026-05-13,FloatingRateCoupon,0.041123,1.016656e+05,1.016656e+05,-9.105133e-08,-8.955960e-11
3,2026-08-13,FloatingRateCoupon,0.039297,1.004266e+05,1.004266e+05,-1.021544e-07,-1.017206e-10
4,2026-11-13,FloatingRateCoupon,0.039251,1.003076e+05,1.003076e+05,-9.547512e-08,-9.518234e-11
5,2027-02-13,FloatingRateCoupon,0.037157,9.495598e+04,9.495598e+04,1.221051e-07,1.285913e-10
6,2027-05-13,FloatingRateCoupon,0.037151,9.184542e+04,9.184542e+04,1.176813e-07,1.281298e-10
7,2027-08-13,FloatingRateCoupon,0.037157,9.495598e+04,9.495598e+04,1.221051e-07,1.285913e-10
8,2027-11-13,FloatingRateCoupon,0.037143,9.492190e+04,9.492190e+04,1.132430e-07,1.193012e-10
9,2028-02-13,FloatingRateCoupon,0.036543,9.338812e+04,9.338812e+04,-2.708985e-07,-2.900781e-10



=== CLP Leg (Receive ICP) ===


,payment_date,type,rate_qs,amount_qs,amount_ore,diff,diff_%
0,2025-11-13,Disbursement,NaN,9.350000e+09,-9.350000e+09,-1.870000e+10,-2.000000e+02
1,2026-02-13,FloatingRateCoupon,0.049768,1.308650e+08,1.308650e+08,-1.245737e-05,-9.519256e-12
2,2026-05-13,FloatingRateCoupon,0.048541,1.237616e+08,1.237616e+08,-8.304417e-05,-6.710011e-11
3,2026-08-13,FloatingRateCoupon,0.047216,1.247661e+08,1.247661e+08,4.981458e-05,3.992639e-11
4,2026-11-13,FloatingRateCoupon,0.047171,1.246602e+08,1.246602e+08,4.775822e-05,3.831071e-11
5,2027-02-13,FloatingRateCoupon,0.045178,1.198986e+08,1.198986e+08,2.086163e-06,1.739939e-12
6,2027-05-13,FloatingRateCoupon,0.045170,1.159693e+08,1.159693e+08,1.490116e-08,1.284923e-14
7,2027-08-13,FloatingRateCoupon,0.045178,1.198986e+08,1.198986e+08,0.000000e+00,0.000000e+00
8,2027-11-13,FloatingRateCoupon,0.045111,1.197378e+08,1.197378e+08,2.086163e-06,1.742275e-12
9,2028-02-13,FloatingRateCoupon,0.042084,1.125047e+08,1.125047e+08,4.187226e-06,3.721825e-12


In [40]:
xccy_swap_rows = []

# Bump ICP (CLP forecast curve)
for tenor, q in zip(icp_tenors, icp_quotes):
    rq = q['rate']
    orig = rq.value()
    rq.setValue(orig + bump)
    v_up = xccy_swap.NPV()
    rq.setValue(orig - bump)
    v_down = xccy_swap.NPV()
    rq.setValue(orig)
    dv01 = (v_up - v_down) / 2
    xccy_swap_rows.append({"Curve": "ICP", "Tenor": tenor, "DV01 (USD/bp)": round(dv01, 2)})

# Bump CLP_COLLUSD (collateral / discount for CLP leg)
for tenor, q in zip(coll_tenors, coll_quotes):
    if 'fxPoints' in q:
        rq = q['fxPoints']
    elif 'rate' in q:
        rq = q['rate']
    else:
        continue
    orig = rq.value()
    rq.setValue(orig + bump)
    v_up = xccy_swap.NPV()
    rq.setValue(orig - bump)
    v_down = xccy_swap.NPV()
    rq.setValue(orig)
    dv01 = (v_up - v_down) / 2
    xccy_swap_rows.append({"Curve": "CLP_COLLUSD", "Tenor": tenor, "DV01 (USD/bp)": round(dv01, 2)})

# Bump SOFR (USD discount)
for tenor, q in zip(sofr_tenors, sofr_quotes):
    rq = q['rate']
    orig = rq.value()
    rq.setValue(orig + bump)
    v_up = xccy_swap.NPV()
    rq.setValue(orig - bump)
    v_down = xccy_swap.NPV()
    rq.setValue(orig)
    dv01 = (v_up - v_down) / 2
    xccy_swap_rows.append({"Curve": "SOFR", "Tenor": tenor, "DV01 (USD/bp)": round(dv01, 2)})

compare_sensitivities(xccy_swap_rows, "Cross-Currency Swap CLP/USD 5Y (receive CLP ICP, pay USD SOFR)")

,Curve,Tenor,DV01 (USD/bp),DV01_QS,Diff
0,ICP,1D,-2.73,-2.7347,0.0047
1,ICP,3M,1.75,1.7523,-0.0023
2,ICP,6M,-2.11,-2.1133,0.0033
3,ICP,1Y,-2.31,-2.3133,0.0033
4,ICP,2Y,-0.57,-0.5737,0.0037
5,ICP,3Y,-1.15,-1.1521,0.0021
6,ICP,5Y,4432.12,4432.1221,-0.0021
7,ICP,7Y,17.16,17.1593,0.0007
8,ICP,10Y,0.00,0.0000,0.0000
9,ICP,20Y,0.00,0.0000,0.0000
